# 09. 스크래치패드(Scratchpad)와 ReAct 비교

> **스크래치패드**는 모델이 최종 답을 내기 전에, 중간 추론 단계를 텍스트로 먼저 적게 하는 기법이다.
> 답만 바로 요구하는 대신 "어떻게 풀지"를 단계별로 쓰게 한 뒤 마지막에 답을 적게 하면,
> 여러 단계를 거쳐야 하는 문제에서 정확도가 올라간다.
> (2021년 Google DeepMind가 제안했고, 최근 Anthropic의 **Thinking Tool**이 같은 아이디어의 현대적 구현이다.)

**이 노트북이 검증하는 것**
- 스크래치패드가 다단계 추론(여러 단계를 순서대로 밟아야 답이 나오는 추론) 정확도를 실제로 올리는가? — 직답 vs `<scratchpad>` 비교
- **ReAct를 대체할 수 있는가?** — 비슷하게 생긴 도구가 여러 개일 때, 어느 방식이 도구를 더 정확히 고르는지 비교
- 스크래치패드는 글이 길어지는 만큼 토큰과 응답 시간을 더 쓴다(테디님 지적) → 정확도와의 맞교환(트레이드오프)을 직접 측정
- Thinking Tool 패턴 — 도구를 호출하기 전에 `think` 단계를 강제로 거치게 하는 방식
- 검토 루프 — 계획을 먼저 받아 사람이 확인한 뒤 실행하는 2단계 방식

*모델: `gpt-5-nano` (temperature 고정, reasoning_effort 지원). 아래 셀을 위에서부터 순서대로 실행하세요.*

In [ ]:
# --- 부트스트랩: 프로젝트 루트를 찾아 research_utils 임포트 ---
import sys
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "research_utils.py").exists():
    _root = _root.parent
sys.path.insert(0, str(_root))

from research_utils import *  # MODEL, OLD_MODEL, ask, ask_meta, ask_json, chat, compare, keyword_hits, get_client

print("모델:", MODEL, "| 구형모델:", OLD_MODEL)
print("API 키 로드됨:", bool(get_client().api_key))

## 왜 스크래치패드가 필요한가?

트랜스포머 모델은 **한 번에 답이 나오는(single-pass) 작업**에는 강하지만, **여러 단계를 순서대로 밟아야 하는 복잡한 문제**에서는 정확도가 크게 떨어지는 경향이 있다. 이때 모델에게 중간 단계를 먼저 텍스트로 적게 하면, 답을 곧장 요구할 때보다 문제를 더 정확히 푼다.

가장 흔한 구조는 다음과 같다.

```
<scratchpad>
  - 답을 내기 전에 어떤 방식으로 풀지 정리
  - 복잡한 문제를 다룰 수 있는 작은 조각으로 분해
  - 틀리기 쉬운 지점 미리 확인
</scratchpad>
<answer> 최종 답 </answer>
```

핵심은 **중간 추론(scratchpad)과 최종 답(answer)을 태그로 나눠 두는 것**이다. 모델이 답을 확정하기 전에 추론 과정을 먼저 쓰도록 형식으로 강제하는 셈이다.

## 실험 1 — 스크래치패드 vs 직답 (다단계 추론)

여러 단계를 거쳐야 답이 나오는 산술 문제를 두 가지 방식으로 풀어 비교한다. (A)는 중간 과정 없이 바로 답만 내게 하고(직답), (B)는 스크래치패드에 중간 계산을 적은 뒤 답을 내게 한다.
**정답은 11**이다. (12 − 3 = 9 → 9 + 9×2 = 27 → 27 − 5 = 22 → 22 ÷ 2 = 11)

In [ ]:
problem = (
    "한 상자에 사과가 12개 있다. 3개를 먹고, 남은 것의 2배를 더 넣은 뒤, "
    "다시 5개를 꺼냈다. 마지막으로 남은 개수를 2명이 똑같이 나누면 한 명당 몇 개인가?"
)

# (A) 직답 강제 — 과정 쓰지 말고 숫자만
a = ask(problem, system="계산 과정을 쓰지 말고 최종 숫자만 한 줄로 답하라.", reasoning_effort="minimal")

# (B) 스크래치패드 강제 — 과정과 답을 태그로 분리
sp_system = (
    "문제를 풀 때 반드시 아래 형식을 지켜라.\n"
    "<scratchpad> 여기에 한 단계씩 중간 계산을 적어라 </scratchpad>\n"
    "<answer> 여기에 최종 숫자만 </answer>"
)
b = ask(problem, system=sp_system, reasoning_effort="minimal")

compare("(A) 직답 (정답 11?)", a, "(B) 스크래치패드 (정답 11?)", b)

## 실험 2 — ReAct vs 스크래치패드 ("ReAct를 대체할 수 있나?")

**ReAct(Reason + Act)**는 "생각 → 행동(도구 호출) → 결과 관찰"을 반복하는 방식이다. 외부 도구를 쓰기 위해 추론을 **행동 중심**으로 짠다. 그런데 도구가 수백 개로 늘고 **성격이 비슷한 도구끼리 겹치면**, 어느 것을 불러야 할지 헷갈려 잘못 고르기 쉽다.

**스크래치패드 방식**은 곧바로 도구를 호출하지 않고, **왜 그 도구를 부르려는지 근거를 먼저 텍스트로 적은 다음** 선택한다. 도구가 많아 혼동되는 상황일수록, 근거를 먼저 정리하게 하면 도구 선택 정확도가 눈에 띄게 올라간다(박사님 지적).

아래는 **서로 헷갈리기 쉬운 금융 도구 6개** 중에서 올바른 것을 고르는 실험이다. 일부러 애매하게 던진 질문으로 두 방식의 차이를 본다.

In [ ]:
tools_desc = """사용 가능한 도구 (성격이 비슷해 혼동하기 쉬움):
- get_savings_rate(bank): 예금(저축) 금리 조회
- get_loan_rate(bank): 대출 금리 조회
- get_fx_rate(pair): 환율 조회
- get_fund_return(fund): 펀드 수익률 조회
- get_card_benefit(card): 카드 혜택 조회
- get_deposit_fx(bank): 외화예금 우대금리 조회"""

# 애매한 질의: '달러 통장'이 외화예금(get_deposit_fx)인지 환율(get_fx_rate)인지 헷갈리게 설계
user_q = "신한은행에서 달러로 통장 만들어서 넣어두면 우대금리 얼마나 쳐줘?"

# (A) ReAct — 짧게 생각하고 바로 Action
react_system = tools_desc + (
    "\n\nReAct 형식으로 답하라. 생각(Thought)은 짧게 한 줄만 하고 곧바로 "
    "Action(도구명과 인자)을 결정하라.\nThought: ...\nAction: 도구명(인자)"
)

# (B) 스크래치패드 우선 — 호출 전 후보별 근거를 먼저 끄적임
scratch_system = tools_desc + (
    "\n\n도구를 고르기 전에 <scratchpad>에 각 후보 도구가 이 질문에 왜 맞는지/틀린지 "
    "근거를 먼저 적어라. 그런 다음 <action>에 최종 도구명과 인자를 적어라."
)

print("### (A) ReAct ###")
print(ask(user_q, system=react_system, reasoning_effort="minimal"))
print("\n" + "=" * 80 + "\n")
print("### (B) 스크래치패드 우선 ###")
print(ask(user_q, system=scratch_system, reasoning_effort="minimal"))

**관찰 포인트**: ReAct는 첫 직관을 따라 `get_fx_rate`(환율)나 `get_savings_rate`(원화 예금)를 성급히 고를 수 있다. 반면 스크래치패드 버전은 후보 도구를 하나씩 따져 보며 "외화 + 예금 = `get_deposit_fx`"라는 정답에 도달할 가능성이 높다.
→ 결국 **추론이 까다롭거나 비슷한 도구가 겹치는 상황에서는, 스크래치패드가 ReAct를 대체하거나 보완**할 수 있다는 것이다.

## 실험 3 — 응답 시간과 토큰의 맞교환 (테디님 지적)

> "스크래치패드를 쓰면 출력이 길어지니 레이턴시(응답이 나오기까지 걸리는 시간)가 늘어나는 것 아니냐?" → **맞다.**
> 중간 추론을 적는 만큼 토큰과 시간을 더 쓴다. 다만 **정확도가 반드시 확보돼야 하는 분야(금융 등)** 에서는 이 비용을 치를 만한 가치가 있다.

`ask_meta`로 같은 문제를 직답 방식과 스크래치패드 방식으로 각각 풀게 하고, **응답 시간과 토큰 사용량을 실제로 측정**한다.

In [ ]:
problem = (
    "한 상자에 사과가 12개 있다. 3개를 먹고, 남은 것의 2배를 더 넣은 뒤, "
    "다시 5개를 꺼냈다. 마지막으로 남은 개수를 2명이 똑같이 나누면 한 명당 몇 개인가?"
)
sp_system = (
    "문제를 풀 때 반드시 아래 형식을 지켜라.\n"
    "<scratchpad> 여기에 한 단계씩 중간 계산을 적어라 </scratchpad>\n"
    "<answer> 여기에 최종 숫자만 </answer>"
)

direct = ask_meta(problem, system="최종 숫자만 답하라.", reasoning_effort="minimal")
scratch = ask_meta(problem, system=sp_system, reasoning_effort="minimal")


def row(label, m):
    print(
        f"{label:14} | 지연 {m['latency']:>5}s | 입력 {m['prompt_tokens']:>4} | "
        f"출력 {m['completion_tokens']:>4} | 추론 {m['reasoning_tokens']}"
    )


print("방식           |     지연 | 입력토큰 |  출력토큰 | 추론토큰")
print("-" * 72)
row("직답", direct)
row("스크래치패드", scratch)
print("\n[답 확인]")
print("  직답       ->", direct["text"])
print("  스크래치패드 ->", scratch["text"])

## 실험 4 — Thinking Tool 패턴 (스크래치패드의 현대적 구현)

Anthropic의 **Think Tool**은 스크래치패드와 같은 아이디어를 도구 형태로 발전시킨 것이다. 모델이 도구를 호출하기 *전에*, 무엇을 하려는지 자신의 계획을 먼저 적게 만든다. 이 실험에서는 JSON으로 `think`(호출 전 계획)를 적은 뒤 `tool`/`args`(실제 호출)로 넘어가도록 형식을 강제해서, 도구를 실행하기 전에 계획을 먼저 쓰게 하는 패턴을 보여 준다.

In [ ]:
import json

tools_desc = """사용 가능한 도구:
- get_savings_rate(bank): 예금 금리
- get_loan_rate(bank): 대출 금리
- get_deposit_fx(bank): 외화예금 우대금리"""

think_system = tools_desc + (
    "\n\n너는 도구를 호출하기 전에 반드시 'think' 단계를 먼저 거친다. "
    "다음 JSON 형식으로만 답하라: "
    '{"think": "호출 전 계획과 근거", "tool": "도구명", "args": {"bank": "..."}}'
)

r = ask_json(
    "농협에서 2년 만기로 5천만원 원화로 예금하면 금리 얼마야?",
    system=think_system,
    reasoning_effort="minimal",
)
print(json.dumps(r, ensure_ascii=False, indent=2))

## 실험 5 — 검토 루프 (계획 먼저 → 사람이 검토 → 실행)

스크래치패드가 특히 유용한 지점은, **모델이 그럴듯하지만 틀린 답을 끝까지 만들어 버리기 전에 잘못된 방향을 미리 잡아낼 수 있다는 것**이다(unite.ai가 강조한 부분). 최종 결과를 곧바로 만들지 않고 계획(스크래치패드)만 먼저 받아 사람이 확인하고, 승인한 뒤에야 실제 결과를 생성하게 하는 2단계 방식이다.

In [ ]:
# 1단계: 계획(스크래치패드)만 받는다 — 최종 답은 아직 쓰지 않게
plan_only = ask(
    "신제품(스마트 물병) 출시 전략이 필요하다. 지금은 계획만 세워라. "
    "<scratchpad>에 접근법, 요소 간 의존성, 정보 격차만 적고 최종 전략은 아직 쓰지 마라.</scratchpad>",
    reasoning_effort="minimal",
)
print("=== 1단계: 계획(스크래치패드)만 ===")
print(plan_only)

# (사람이 계획을 검토하고 승인했다고 가정) 2단계: 승인된 계획대로 실행
final = ask(
    "아래 계획을 검토했고 승인한다. 이 계획을 그대로 따라 최종 출시 전략을 작성하라.\n\n[승인된 계획]\n"
    + plan_only,
    reasoning_effort="minimal",
)
print("\n=== 2단계: 승인 후 최종 전략 ===")
print(final)

## 요약 — 언제 무엇을 쓸까?

| 상황 | 추천 | 이유 |
|---|---|---|
| 단순하고 빠른 작업, 써야 할 도구가 하나로 분명할 때 | **ReAct** | 바로 행동하는 편이 빠르고 효율적 |
| 여러 단계를 거치는 추론, 도구가 많고 서로 비슷해 겹칠 때 | **스크래치패드 / Thinking Tool** | 도구를 부르기 전에 근거를 먼저 적어 정확도를 높임 |
| 정확도가 반드시 필요한 분야(금융 등) | **스크래치패드** | 응답 시간을 더 쓰더라도 정확성이 우선 |
| 틀린 결과가 큰 손해로 이어지는 작업 | **검토 루프** | 계획을 먼저 검토해 오답을 조기에 차단 |

**핵심**: 스크래치패드는 토큰과 응답 시간을 더 쓰지만(테디님 지적), 그 대신 모델이 답을 확정하기 전에 중간 추론을 거치게 해 정확도를 끌어올린다. 단순한 작업에는 ReAct가 낫고, **추론이 복잡하거나 도구 호출이 잦고 헷갈리는 상황에서는 스크래치패드가 ReAct를 대체하거나 보완**한다. 이것이 현대 프롬프트 엔지니어링의 핵심 전략 중 하나다.